# U4-03 · Series temporales, textos y autocodificadores

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 4*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894
> Datos: Almeida, T. A., Gómez Hidalgo, J. M. y Yamakami, A. (2011). SMS Spam Collection, UCI, CC BY 4.0. https://doi.org/10.1145/2034691.2034742
> Datos: BloodMNIST (MedMNIST v2, CC BY 4.0). Yang, J. et al. (2023). *Scientific Data*, 10, 41. https://doi.org/10.1038/s41597-022-01721-8, a partir de Acevedo, A. et al. (2020). *Data in Brief*, 30, 105474. https://doi.org/10.1016/j.dib.2020.105474. No apto para uso clínico.

In [ ]:
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["hour.csv", "ventanas.py", "sms2.tsv", "sms.py", "bloodmnist.npz", "sangre.py", "judias.csv", "keras_judias.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://machinelearning-formacion.github.io/ifcd107/recursos/datos/u4/" + f, f)

## 4.4 · Redes recurrentes: prever la demanda

`o1_referencias.py`

In [ ]:
import numpy as np
from ventanas import X, y, ent, pru, real, idx, h

print("ventanas:", X.shape, "| entrenamiento", ent.sum(),
      "| prueba", pru.sum())
eam = lambda p: np.abs(real[pru] - p).mean()
cnt = h.cnt.to_numpy()
print(f"la hora anterior:      EAM {eam(cnt[idx - 1][pru]):5.1f}")
ayer = np.expm1(X[pru, 0, 0])              # hace 24 horas
print(f"la misma hora de ayer: EAM {eam(ayer):5.1f}")

`o2_recurrente.py`

In [ ]:
import time
import numpy as np
import keras
from keras import layers
from ventanas import X, y, ent, pru, real

def probar(nombre, capas):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([keras.Input((24, 5))] + capas
                         + [layers.Dense(1)])
    m.compile(keras.optimizers.Adam(0.003), "mae")
    t = time.perf_counter()
    m.fit(X[ent], y[ent], epochs=25, batch_size=64, verbose=0,
          validation_split=0.15, callbacks=[
              keras.callbacks.EarlyStopping(
                  patience=4, restore_best_weights=True)])
    p = np.expm1(m.predict(X[pru], verbose=0).ravel())
    print(f"{nombre:<10} parámetros {m.count_params():6d}"
          f"  EAM {np.abs(real[pru] - p).mean():5.1f}"
          f"  ({time.perf_counter() - t:.0f} s)")
probar("densa", [layers.Flatten(),
                 layers.Dense(64, activation="relu")])
probar("RNN simple", [layers.SimpleRNN(64)])
probar("LSTM", [layers.LSTM(64)])
probar("GRU", [layers.GRU(64)])

`o4_memoria.py`

In [ ]:
import numpy as np
import keras
from keras import layers
from ventanas import serie, exo, t, h

def ventanas(L):
    idx = np.arange(168, len(serie))      # mismas horas para todas
    ok = t.to_numpy()[idx] - t.to_numpy()[idx - 168] == \
        np.timedelta64(168, "h")
    idx = idx[ok]
    X = np.stack([np.column_stack([serie[i - L:i],
                                   exo[i - L + 1:i + 1]])
                  for i in idx])
    ent = h.dteday.to_numpy()[idx] < np.datetime64("2012-07-01")
    return X, serie[idx], ent, h.cnt.to_numpy()[idx]
for L in [6, 24, 72, 168]:
    X, y, ent, real = ventanas(L)
    keras.utils.set_random_seed(0)
    m = keras.Sequential([keras.Input((L, 5)), layers.GRU(64),
                          layers.Dense(1)])
    m.compile(keras.optimizers.Adam(0.003), "mae")
    m.fit(X[ent], y[ent], epochs=25, batch_size=64, verbose=0,
          validation_split=0.15, callbacks=[
              keras.callbacks.EarlyStopping(
                  patience=4, restore_best_weights=True)])
    p = np.expm1(m.predict(X[~ent], verbose=0).ravel())
    print(f"ventana de {L:3d} horas: EAM"
          f" {np.abs(real[~ent] - p).mean():5.1f}")

`o3_horizonte.py`

In [ ]:
import numpy as np
import keras
from keras import layers
from ventanas import X, serie, idx, ent, pru, t

H = 24                                    # predecir 24 horas de golpe
ok = idx + H <= len(serie)
ok[ok] = (t.to_numpy()[idx[ok] + H - 1] - t.to_numpy()[idx[ok]]
          == np.timedelta64(H - 1, "h"))  # 24 horas seguidas
Y = np.stack([serie[i:i + H] for i in idx[ok]])
Xo, eo, po = X[ok], ent[ok], pru[ok]
keras.utils.set_random_seed(0)
m = keras.Sequential([keras.Input((24, 5)), layers.GRU(64),
                      layers.Dense(H)])
m.compile(keras.optimizers.Adam(0.003), "mae")
m.fit(Xo[eo], Y[eo], epochs=25, batch_size=64, verbose=0,
      validation_split=0.15, callbacks=[keras.callbacks.EarlyStopping(
          patience=4, restore_best_weights=True)])
P = np.expm1(m.predict(Xo[po], verbose=0)); R = np.expm1(Y[po])
eam = np.abs(P - R).mean(0)
print("ventanas de prueba:", po.sum())
print("EAM según la distancia al presente:")
for k in [1, 2, 3, 6, 12, 24]:
    print(f"  {k:2d} h: {eam[k - 1]:5.1f}")

## 4.5 · Redes para texto: mensajes no deseados

`p1_mensajes.py`

In [ ]:
import re
from collections import Counter
from sms import d

print(d.shape, "| spam:", d.spam.sum(), f"({d.spam.mean():.1%})")
print("repetidos:", d.texto.duplicated().sum())
d["largo"] = d.texto.str.len()
print(d.groupby("clase").largo.median().to_string())
pal = lambda c: Counter(w for t in d.texto[d.clase == c]
                        for w in re.findall(r"[a-z£]+", t.lower()))
spam, ham = pal("spam"), pal("ham")
raras = {w: n for w, n in spam.items() if n >= 30 and ham[w] < 5}
print("palabras de spam casi ausentes en el resto:",
      sorted(raras, key=raras.get, reverse=True)[:10])

`p2_bolsa.py`

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
from sms import t_e, t_p, y_e, y_p

m = make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2)),
                  LogisticRegression(C=10, max_iter=2000))
m.fit(t_e, y_e)
print("vocabulario:", len(m[0].vocabulary_), "términos")
print(classification_report(y_p, m.predict(t_p), digits=3,
                            target_names=["normal", "spam"]))
w = m[-1].coef_[0]
voc = m[0].get_feature_names_out()
print("más spam:", list(voc[np.argsort(w)[-8:][::-1]]))
print("menos spam:", list(voc[np.argsort(w)[:8]]))

`p3_embedding.py`

In [ ]:
import keras
from keras import layers
from sklearn.metrics import f1_score
from sms import t_e, t_p, y_e, y_p

def probar(tokens, tipo):
    keras.utils.set_random_seed(0)
    vec = layers.TextVectorization(max_tokens=tokens,
                                   output_sequence_length=60)
    vec.adapt(t_e.to_numpy())
    Xe, Xp = vec(t_e.to_numpy()), vec(t_p.to_numpy())
    if tipo == "media":                   # media de los vectores
        capas = [layers.Embedding(tokens, 16),
                 layers.GlobalAveragePooling1D()]
    else:                                 # detector de 3 palabras
        capas = [layers.Embedding(tokens, 32),
                 layers.Conv1D(64, 3, activation="relu"),
                 layers.GlobalMaxPooling1D()]
    m = keras.Sequential([keras.Input((60,), dtype="int64")] + capas
                         + [layers.Dropout(0.3),
                            layers.Dense(1, activation="sigmoid")])
    m.compile("adam", "binary_crossentropy")
    m.fit(Xe, y_e.to_numpy(), epochs=10, batch_size=32, verbose=0)
    p = (m.predict(Xp, verbose=0).ravel() > 0.5).astype(int)
    print(f"vocabulario {tokens:4d}, {tipo:<12}"
          f" acierto {(p == y_p).mean():.3f}"
          f"  F1 spam {f1_score(y_p, p):.3f}")
    return vec
vec = probar(8000, "media")
print("«Free entry now» ->", vec(["Free entry now"]).numpy()[0, :5])
for tokens, tipo in [(8000, "convolución"), (1000, "media"),
                     (1000, "convolución")]:
    probar(tokens, tipo)

`p7_bilstm.py`

In [ ]:
import keras
from keras import layers
from sklearn.metrics import f1_score
from sms import t_e, t_p, y_e, y_p

keras.utils.set_random_seed(0)
vec = layers.TextVectorization(max_tokens=1000,
                               output_sequence_length=60)
vec.adapt(t_e.to_numpy())
Xe, Xp = vec(t_e.to_numpy()), vec(t_p.to_numpy())
m = keras.Sequential([
    keras.Input((60,), dtype="int64"),
    layers.Embedding(1000, 32),
    layers.Bidirectional(layers.LSTM(32)),   # izquierda y derecha
    layers.Dropout(0.3), layers.Dense(1, activation="sigmoid")])
m.compile("adam", "binary_crossentropy")
m.fit(Xe, y_e.to_numpy(), epochs=10, batch_size=32, verbose=0)
p = (m.predict(Xp, verbose=0).ravel() > 0.5).astype(int)
print(f"parámetros {m.count_params()}:"
      f" acierto {(p == y_p).mean():.3f}"
      f"  F1 spam {f1_score(y_p, p):.3f}")

`p6_subpalabras.py`

In [ ]:
import re
from collections import Counter
from sms import t_e

frec = Counter(w for t in t_e for w in re.findall(r"[a-z]+",
                                                  t.lower()))
voc = {tuple(w) + ("_",): n for w, n in frec.items()}  # letras
def unir(voc):                            # un paso del algoritmo BPE
    pares = Counter()
    for pal, n in voc.items():
        for a, b in zip(pal, pal[1:]):
            pares[a, b] += n
    (a, b), n = pares.most_common(1)[0]
    nuevo = {}
    for pal, k in voc.items():
        p, i = [], 0
        while i < len(pal):
            if pal[i:i + 2] == (a, b):
                p.append(a + b); i += 2
            else:
                p.append(pal[i]); i += 1
        nuevo[tuple(p)] = k
    return nuevo, a + b, n
fusiones = []
for paso in range(300):
    voc, f, n = unir(voc)
    fusiones.append(f)
print("primeras fusiones:", fusiones[:10])
print("fusiones 291 a 300:", fusiones[-10:])
def trocear(w):
    p = list(w) + ["_"]
    for f in fusiones:                    # aplicar en el mismo orden
        i = 0
        while i < len(p) - 1:
            if p[i] + p[i + 1] == f:
                p[i:i + 2] = [f]
            else:
                i += 1
    return p
for w in ["congratulations", "unbelievable", "smartphone"]:
    print(w, "->", trocear(w))

`p4_atencion.py`

In [ ]:
import keras
from keras import layers, ops
from sklearn.metrics import f1_score
from sms import t_e, t_p, y_e, y_p

keras.utils.set_random_seed(0)
vec = layers.TextVectorization(max_tokens=1000,
                               output_sequence_length=60)
vec.adapt(t_e.to_numpy())
Xe, Xp = vec(t_e.to_numpy()), vec(t_p.to_numpy())

class Posicion(layers.Layer):           # palabra + lugar que ocupa
    def __init__(self, n, d, **kw):
        super().__init__(**kw)
        self.pal = layers.Embedding(1000, d)
        self.pos = layers.Embedding(n, d)
    def call(self, x):
        return self.pal(x) + self.pos(ops.arange(ops.shape(x)[1]))

ent = keras.Input((60,), dtype="int64")
x = Posicion(60, 32)(ent)
a = layers.MultiHeadAttention(num_heads=2, key_dim=16)(x, x)
x = layers.LayerNormalization()(x + a)          # atención + atajo
f = layers.Dense(64, activation="relu")(x)
x = layers.LayerNormalization()(x + layers.Dense(32)(f))
x = layers.GlobalMaxPooling1D()(x)
sal = layers.Dense(1, activation="sigmoid")(layers.Dropout(0.3)(x))
m = keras.Model(ent, sal)
m.compile("adam", "binary_crossentropy")
m.fit(Xe, y_e.to_numpy(), epochs=10, batch_size=32, verbose=0)
p = (m.predict(Xp, verbose=0).ravel() > 0.5).astype(int)
print(f"parámetros {m.count_params()}:"
      f" acierto {(p == y_p).mean():.3f}"
      f"  F1 spam {f1_score(y_p, p):.3f}")

`p5_generar.py`

In [ ]:
import numpy as np
import keras
from keras import layers
from sms import u

texto = "\n".join(u.texto[u.spam == 1]).lower()
texto = "".join(c for c in texto if c.isascii() or c == "£")
car = sorted(set(texto)); ind = {c: i for i, c in enumerate(car)}
cod = np.array([ind[c] for c in texto])
print(f"{len(texto)} caracteres, {len(car)} distintos")
L = 40                                    # leer 40, predecir el 41
X = np.stack([cod[i:i + L] for i in range(0, len(cod) - L, 3)])
y = np.array([cod[i + L] for i in range(0, len(cod) - L, 3)])
keras.utils.set_random_seed(0)
m = keras.Sequential([keras.Input((L,)),
                      layers.Embedding(len(car), 32),
                      layers.GRU(256),
                      layers.Dense(len(car), activation="softmax")])
m.compile("adam", "sparse_categorical_crossentropy")
h = m.fit(X, y, epochs=20, batch_size=128, verbose=0)
print(f"pérdida: época 1 {h.history['loss'][0]:.2f},"
      f" época 20 {h.history['loss'][-1]:.2f}")
rng = np.random.default_rng(0)
def escribir(inicio, n, temperatura):
    s = inicio
    for _ in range(n):
        x = np.array([[ind[c] for c in s[-L:].rjust(L, " ")]])
        p = np.log(m.predict(x, verbose=0)[0] + 1e-9) / temperatura
        p = np.exp(p - p.max()); p /= p.sum()
        s += car[rng.choice(len(car), p=p)]
    return s
for temp in [0.3, 1.0]:
    print(f"temperatura {temp}:",
          escribir("congratulations! you have won ", 50, temp))

## 4.6 · Autocodificadores y modelos generativos

`q1_autocodificador.py`

In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt
from keras import layers
from sklearn.linear_model import LogisticRegression
from sangre import X_e, y_e, X_p, y_p

keras.utils.set_random_seed(0)
cod = keras.Sequential([                  # 2352 números -> 32
    keras.Input((28, 28, 3)),
    layers.Conv2D(32, 3, 2, "same", activation="relu"),   # 14x14
    layers.Conv2D(64, 3, 2, "same", activation="relu"),   # 7x7
    layers.Flatten(), layers.Dense(32)])
dec = keras.Sequential([                  # 32 -> 28x28x3
    keras.Input((32,)), layers.Dense(7 * 7 * 64, activation="relu"),
    layers.Reshape((7, 7, 64)),
    layers.Conv2DTranspose(32, 3, 2, "same", activation="relu"),
    layers.Conv2DTranspose(3, 3, 2, "same", activation="sigmoid")])
ae = keras.Sequential([cod, dec])
ae.compile("adam", "mse")
h = ae.fit(X_e, X_e, epochs=15, batch_size=128, verbose=0,
           validation_data=(X_p, X_p))
print(f"error cuadrático medio por píxel: entrenamiento"
      f" {h.history['loss'][-1]:.4f}, prueba"
      f" {h.history['val_loss'][-1]:.4f}")
print("compresión:", 28 * 28 * 3, "->", 32, "números")
Z_e = cod.predict(X_e, verbose=0); Z_p = cod.predict(X_p, verbose=0)
lr = LogisticRegression(max_iter=3000).fit(Z_e, y_e)
print(f"regresión logística sobre los 32 códigos: acierto"
      f" {lr.score(Z_p, y_p):.3f}")
R = ae.predict(X_p[:8], verbose=0)
fig, ejes = plt.subplots(2, 8, figsize=(4.6, 1.2))
for i in range(8):
    ejes[0, i].imshow(X_p[i]); ejes[1, i].imshow(R[i])
    ejes[0, i].axis("off"); ejes[1, i].axis("off")
plt.show()
ae.save("autocodificador.keras")

`q2_rareza.py`

In [ ]:
import numpy as np
import pandas as pd
import keras
from sangre import X_e, X_p, y_p, nombres

ae = keras.models.load_model("autocodificador.keras")
def error(X):
    return ((ae.predict(X, verbose=0) - X) ** 2).mean((1, 2, 3))
umbral = np.quantile(error(X_e), 0.99)    # el 1 % más raro
e = error(X_p)
d = pd.DataFrame({"clase": np.array(nombres)[y_p], "error": e,
                  "rara": e > umbral})
print(f"umbral: {umbral:.4f} | imágenes de prueba marcadas:"
      f" {d.rara.sum()} de {len(d)}")
print(d.groupby("clase").agg(error_medio=("error", "mean"),
                             raras=("rara", "sum"))
      .sort_values("error_medio").round(4).to_string())

`q3_vae.py`

In [ ]:
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
from keras import layers
from sangre import X_e

keras.utils.set_random_seed(0)
d = 16                                    # dimensión del código
cod = keras.Sequential([
    keras.Input((28, 28, 3)),
    layers.Conv2D(32, 3, 2, "same", activation="relu"),
    layers.Conv2D(64, 3, 2, "same", activation="relu"),
    layers.Flatten(), layers.Dense(2 * d)])   # media y log-varianza
dec = keras.Sequential([
    keras.Input((d,)), layers.Dense(7 * 7 * 64, activation="relu"),
    layers.Reshape((7, 7, 64)),
    layers.Conv2DTranspose(32, 3, 2, "same", activation="relu"),
    layers.Conv2DTranspose(3, 3, 2, "same", activation="sigmoid")])
opt = keras.optimizers.Adam(1e-3)
datos = tf.data.Dataset.from_tensor_slices(
    X_e.astype("float32")).shuffle(12000, seed=0).batch(128)
for epoca in range(1, 21):
    for x in datos:
        with tf.GradientTape() as cinta:
            mu, logv = tf.split(cod(x), 2, axis=1)
            ruido = tf.random.normal(tf.shape(mu))
            z = mu + tf.exp(0.5 * logv) * ruido   # muestrear
            r = dec(z)
            rec = tf.reduce_sum(tf.square(x - r), [1, 2, 3])
            kl = -0.5 * tf.reduce_sum(
                1 + logv - mu**2 - tf.exp(logv), 1)
            L = tf.reduce_mean(rec + kl)
        pesos = cod.trainable_weights + dec.trainable_weights
        opt.apply_gradients(zip(cinta.gradient(L, pesos), pesos))
    if epoca in (1, 20):
        print(f"época {epoca:2d}: reconstrucción"
              f" {float(tf.reduce_mean(rec)):.1f},"
              f" KL {float(tf.reduce_mean(kl)):.1f}")
nuevas = dec(np.random.default_rng(0).normal(size=(16, d))).numpy()
print("imágenes generadas:", nuevas.shape)
fig, ejes = plt.subplots(2, 8, figsize=(4.6, 1.2))
for e, img in zip(ejes.flat, nuevas):
    e.imshow(img); e.axis("off")
plt.show()

## 4.7 · Ajustar y poner en producción

`r1_busqueda.py`

In [ ]:
import numpy as np
import keras
from keras_judias import X_e, X_p, y_e, y_p

rng = np.random.default_rng(0)
X_a, y_a = X_e[:8000], y_e[:8000]          # ajuste
X_v, y_v = X_e[8000:], y_e[8000:]          # validación
res = []
for prueba in range(12):                   # 12 combinaciones al azar
    q = {"capas": int(rng.integers(1, 4)),
         "neuronas": int(rng.choice([16, 64, 256])),
         "dropout": float(rng.choice([0.0, 0.2, 0.5])),
         "tasa": float(10 ** rng.uniform(-4, -2))}
    keras.utils.set_random_seed(0)
    capas = [keras.Input(shape=(16,))]
    for _ in range(q["capas"]):
        capas += [keras.layers.Dense(q["neuronas"],
                                     activation="relu"),
                  keras.layers.Dropout(q["dropout"])]
    m = keras.Sequential(capas + [keras.layers.Dense(
        7, activation="softmax")])
    m.compile(keras.optimizers.Adam(q["tasa"]),
              "sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(X_a, y_a, epochs=20, batch_size=128, verbose=0)
    res.append((m.evaluate(X_v, y_v, verbose=0)[1], q, m))
res.sort(key=lambda r: r[0], reverse=True)
for a, q, _ in res[:3] + res[-2:]:
    print(f"val. {a:.3f}  capas {q['capas']} x {q['neuronas']:3d},"
          f" dropout {q['dropout']}, tasa {q['tasa']:.5f}")
mejor = res[0][2].evaluate(X_p, y_p, verbose=0)[1]
print(f"mejor en la prueba: {mejor:.3f}")

**Prueba.** En `p3_embedding.py`, prueba un vocabulario de 3000 términos con la convolución. ¿Dónde queda su F1?

**Ejercicio.** En `p5_generar.py`, empieza la frase con otro texto y prueba una temperatura de 0,6.